In [4]:
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances
from sklearn_extra.cluster._k_medoids import KMedoids

from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from scipy.spatial.distance import cdist, pdist, squareform
from scipy.sparse.csgraph import minimum_spanning_tree, connected_components

from scipy.cluster.hierarchy import linkage, fcluster


In [6]:

# Load sample dataset (or your custom data X)
iris = load_iris()
X = iris.data
k = 3  # Number of clusters

# ==========================================
# 1. FUNCTION: Calculate Medoids Total Cost
# ==========================================
def calculate_medoid_cost(X, labels, medoid_indices):
    """Calculates the sum of absolute distances from each point to its assigned medoid."""
    dist_matrix = euclidean_distances(X, X[medoid_indices])
    min_distances = dist_matrix[np.arange(len(X)), labels]
    return np.sum(min_distances)

# ==========================================
# 2. RUN K-MEDOIDS (Heuristic / Exact reference)
# ==========================================
kmedoids = KMedoids(n_clusters=k, random_state=42).fit(X)
kmed_labels = kmedoids.labels_
kmed_indices = kmedoids.medoid_indices_

# Calculate objective function cost for K-Medoids
cost_kmed = calculate_medoid_cost(X, kmed_labels, kmed_indices)

# ==========================================
# 3. RUN MST-BASED HEURISTIC CLUSTERING
# ==========================================
# Using single-linkage hierarchical clustering (closely related to MST cutting)
Z = linkage(X, method='single')
mst_labels = fcluster(Z, t=k, criterion='maxclust') - 1

# ==========================================
# 4. DIRECT COMPARISON (K-Medoids vs MST)
# ==========================================
# Partition similarity metrics
ari = adjusted_rand_score(kmed_labels, mst_labels)
nmi = normalized_mutual_info_score(kmed_labels, mst_labels)

print("=== COMPARISON RESULTS: K-MEDOIDS vs MST ===")
print(f"K-Medoids Objective Cost : {cost_kmed:.4f}")
print(f"Adjusted Rand Index (ARI): {ari:.4f}  (1.0 = identical partitions)")
print(f"Normalized Mutual Info (NMI): {nmi:.4f}")

=== COMPARISON RESULTS: K-MEDOIDS vs MST ===
K-Medoids Objective Cost : 98.8686
Adjusted Rand Index (ARI): 0.5798  (1.0 = identical partitions)
Normalized Mutual Info (NMI): 0.7220


Objective cost = total sum of distances of each point to its medoid.


In [7]:
ari

0.5797888001588342

In [ ]:

# Exemple : Comparer AMPL et K-Means
ari_kmeans = adjusted_rand_index(y_ampl, y_kmeans)
nmi_kmeans = normalized_mutual_info_score(y_ampl, y_kmeans)

# Exemple : Comparer AMPL et MST
ari_mst = adjusted_rand_index(y_ampl, y_mst)
nmi_mst = normalized_mutual_info_score(y_ampl, y_mst)

print(f"Comparaison AMPL vs K-Means -> ARI: {ari_kmeans:.3f}, NMI: {nmi_kmeans:.3f}")
print(f"Comparaison AMPL vs MST    -> ARI: {ari_mst:.3f}, NMI: {nmi_mst:.3f}")